In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime


print("🥈 TRANSFORMATION BRONZE → SILVER")

df_bronze = spark.table("water_catalog.bronze.water_quality")

print(f"📥 Bronze : {df_bronze.count():,} lignes")
df_bronze.printSchema()

In [0]:
print([ (c, t) for c, t in df_bronze.dtypes if t == "void"])

In [0]:
def nettoyer_colonne_numerique(col):
    return F.regexp_replace(
        F.regexp_extract(col, r"([0-9]+[.,]?[0-9]*)", 1),
        ",",
        "."
    ).cast(DoubleType())

In [0]:
print("🔧 Transformation Bronze → Silver")

df_silver = df_bronze

# ✅ Nettoyage des colonnes qualité
df_silver = df_silver.withColumn(
    "limite_qualite",
    nettoyer_colonne_numerique(F.col("limite_qualite_parametre"))
)

if "reference_qualite_parametre" in df_silver.columns:
    df_silver = df_silver.withColumn(
        "reference_qualite",
        nettoyer_colonne_numerique(F.col("reference_qualite_parametre"))
    )

# ✅ sécurisation résultat numérique
df_silver = df_silver.withColumn(
    "resultat_numerique",
    F.col("resultat_numerique").cast(DoubleType())
)

# ✅ ajout colonne ingestion
df_silver = df_silver.withColumn(
    "ingestion_date",
    F.current_timestamp()
)

# ✅ suppression colonnes inutiles
cols_to_drop = []

if "libelle_parametre_web" in df_silver.columns:
    cols_to_drop.append("libelle_parametre_web")

if "reference_qualite_parametre" in df_silver.columns:
    cols_to_drop.append("reference_qualite_parametre")

df_silver = df_silver.drop(*cols_to_drop)

print("✅ Transformation terminée")


In [0]:
from pyspark.sql import functions as F

def gerer_nulls(df):

    print("🧹 Traitement des NULL...")

    # 1. Colonnes critiques → on SUPPRIME les lignes si null
    df = df.filter(
        F.col("nom_commune").isNotNull() &
        F.col("code_departement").isNotNull() &
        F.col("resultat_numerique").isNotNull() &
        F.col("date_prelevement").isNotNull()
    )

    # 2. Colonnes numériques → remplacement par 0 ou moyenne selon cas
    df = df.fillna({
        "limite_qualite": 0,
        "reference_qualite": 0
        
    })

    # 3. Colonnes texte → valeur "UNKNOWN"
    df = df.fillna({
        "libelle_parametre": "UNKNOWN",
        "code_parametre_cas": "UNKNOWN",
        "limite_qualite_parametre" :0
    })

    return df

In [0]:
from pyspark.sql import functions as F

def transformer_dates(df):

    print("📅 Transformation des dates...")

    # 1. conversion en DATE
    df = df.withColumn(
        "date_prelevement",
        F.to_date(F.col("date_prelevement"))
    )

    # 2. sécurisation (si format bizarre)
    df = df.filter(F.col("date_prelevement").isNotNull())

    # 3. extraction année / mois
    df = df.withColumn(
        "annee_prelevement",
        F.year("date_prelevement")
    ).withColumn(
        "mois_prelevement",
        F.month("date_prelevement")
    )

    # 4. option utile (très pro pour analytics)
    df = df.withColumn(
        "trimestre_prelevement",
        F.quarter("date_prelevement")
    )

    return df

In [0]:
df_silver= transformer_dates(df_silver)
df_silver = gerer_nulls(df_silver)
df_silver.display()

In [0]:
print("💾 Écriture Silver")

df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("water_catalog.silver.water_quality_clean")

print("✅ Table Silver créée")